In [ ]:
# =================================
# DIABETES PREDICTION — ML PIPELINE
# =================================
# Dataset: Pima Indians Diabetes
# Goal: Predict diabetes (Yes/No)
# Models: LR, DT, RF, KNN, NB, MLP
# Best Model: Logistic Regression
# Accuracy: 77.22%
# Author: Sai
# =================================

In [ ]:
#EDA

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

data = pd.read_csv("https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv")

#EDA

# PLOTTING THE HEAT MAP TO FIND THE RELATIONS BETWEEN DATASET COLUMNS

corr = data.corr()
sns.heatmap(corr , annot=True , cmap="coolwarm" , fmt="0.2f")
plt.xlabel("relation")
plt.ylabel("relation")
plt.title("Diabetes correlation Heatmap")
plt.show()

# FOUND THAT DIABETES DEPENDS MOSTLY ON INSULIN , AGE , BMI

# NOW LET US SEE THINGS ON HISTPLOT

sns.histplot(data,x="Outcome" ,kde=True, color="skyblue")
plt.show()

In [ ]:
# Finding the best model , for diabeties analysis.

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB



data = pd.read_csv("https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv")

X = data.drop("Outcome" , axis=1)
Y = data["Outcome"]

data.isnull().sum() # NO NULL VALUES IN DATA SET

X_train, X_test,Y_train, Y_test = train_test_split(X,Y,test_size=0.2, random_state=42)

scalar = StandardScaler() # to scale the columns which are very far as per distance 
x_scaled = scalar.fit_transform(X) # scaling process

k_values = range(3, 31)
cv_scores = []

# KNN

model1 = KNeighborsClassifier(n_neighbors=17)
kn_scores = cross_val_score(model1, x_scaled, Y, cv=5, scoring="accuracy")# cross validation score for best k neighbours

# NAIVE BAYES

nb_model = GaussianNB()
scores_nb = cross_val_score(nb_model, X, Y, cv=5)

# NEURAL NETWORK

X_1 = data.drop("Outcome" , axis=1).values # ONLY FOR NEURAL NETWORK WE NEED VALUES
Y = data["Outcome"]


scalar = StandardScaler() # to scale the columns which are very far as per distance 
x_scaled_neu = scalar.fit_transform(X_1) # scaling process


model2 = MLPClassifier(
    hidden_layer_sizes=(16, 16),
    activation='relu',
    max_iter=500,
    random_state=42
)

neu_scores = cross_val_score(model2, x_scaled_neu, Y , cv=5, scoring="accuracy")


models = {
    # "Linear regression" : LinearRegression(max_iter=1000),
    "Logistic regression" : LogisticRegression(max_iter=1000),
    "Decision tree" : DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random forest" : RandomForestClassifier(n_estimators=50, random_state=42),
}

# SCORE CALCULATION

for name, model in models.items():
    scores = cross_val_score(model,X, Y, cv=5,scoring="accuracy")
    print(f"---{name}---")
    print(f"mean accuracy   : {scores.mean() :.4f}")
    print(f"Std deviation   : {round(scores.std(), 4)}\n")




print(f"---KNN___")
print("Mean accuracy  : ", round(kn_scores.mean(), 4))
print(f"Std deviation : {round(kn_scores.std(), 4)}\n")

print("---NAIVE BAYES---")
print("mean accuracy  :  ", round(scores_nb.mean(), 4))
print("Std deviation  :", round(scores_nb.std(), 4))
print()

print("---Neural Network---")
print("mean accuracy  : ", round(neu_scores.mean(), 4))
print("Std deviation  :", round(neu_scores.std(), 4))



""" BEST MODEL RANDOM FOREST FOR LARGE DATA SETS
SINCE THE DATA IS OF 768 ROWS WE ARE CHOSING LOGISTIC REGRESSION BECAUSE OF LESS STANTDARD DEVIATION"""

model_end = LogisticRegression(max_iter=1000)
model_end.fit(X_train, Y_train)
predictions = model_end.predict(X_test)

# Full report 
print(classification_report(Y_test, predictions))


In [ ]:
# TO FIND THE NUMBER OF best ESTIMATORS FOR RANDOM FOREST

import matplotlib.pyplot as plt

estm = range(10, 100)
score_trees = []

for est in estm:
    model = RandomForestClassifier(n_estimators=est, random_state=42)
    scores = cross_val_score(model,X, Y, cv=5,scoring="accuracy")

    score_trees.append(scores.mean())

    
plt.plot(estm, score_trees, color="blue", linestyle=":", label="finding the no of tress")
plt.xlabel("number of tress")
plt.ylabel("scores per tree")
plt.xticks(range(10, 101, 10))
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6) # Added gridlines to make it easier to read
plt.show()

In [ ]:
# TO FIND THE BEST HYPER PARAMETER (tuning)

import pandas as pd

data = pd.read_csv("https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv")

X = data.drop("Outcome" , axis=1)
Y = data["Outcome"]
data.isnull().sum()

# You used max_depth=6 blindly
# Should have found best depth first:

for depth in [3, 4, 5, 6, 7]:
    dt = DecisionTreeClassifier(
        max_depth=depth, random_state=42)
    scores = cross_val_score(dt, X, Y, cv=5)
    print(f"depth={depth} → {scores.mean():.4f}")